# 第3回：pandasで表データに触る

**今日の問い：初めて見る表データを受け取ったら、最初に何を見るか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 初見データの形・型・欠損・要約統計を確認する
- locで列と条件を明示して抽出する
- groupbyとaggで比較表を作る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- DataFrame：行と列を持つ表
- Series：DataFrameの1列に相当するデータ
- 欠損値：未測定・不明など値が存在しない状態
- 集約：複数行を件数や平均などへまとめる処理

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：最初の健康診断

行数・列数、列名、データ型、欠損数を順番に確認します。


In [ ]:
print("形:", df.shape)
display(pd.DataFrame({"データ型": df.dtypes, "欠損数": df.isna().sum()}))
display(df.select_dtypes(include="number").describe().T)


## 行と列を選ぶ


In [ ]:
columns = ["sample_id", "solvent", "temperature_c", "yield_pct", "active"]
display(df.loc[:4, columns])
high_yield = df.loc[df["yield_pct"] >= 60, columns]
print("収率60%以上:", len(high_yield), "件")
high_yield.head()


## TRY：カテゴリごとに比べる


In [ ]:
solvent_summary = (
    df.groupby("solvent", dropna=False)
      .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"), 活性率=("active", "mean"))
      .sort_values("平均収率", ascending=False)
)
solvent_summary.round(2)


## CHANGE

`solvent`を`catalyst`または`scaffold_group`へ変えます。順位が変わる理由をデータだけから断定せず、仮説として書きます。

## CHALLENGE

`query`または複数条件を使い、「Cat-Aかつ温度80度以上」の行を抽出します。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- 平均だけでなく件数とばらつきを一緒に見る
- 行番号とsample_idを混同しない
- カテゴリ別の差は因果関係とは限らない


In [ ]:
quality_report = pd.DataFrame({
    "型": df.dtypes.astype(str),
    "欠損率": df.isna().mean(),
    "ユニーク数": df.nunique(dropna=True),
})
display(quality_report.sort_values("欠損率", ascending=False).head(10).round(3))
display(pd.pivot_table(df, index="catalyst", columns="solvent", values="yield_pct", aggfunc=["count", "mean"]).round(1))


## よくある誤り

- 列の単位や定義を確認せず計算する
- 欠損行を知らないまま自動で落とす
- 件数が極端に少ない群の平均を強く信じる

## SELF-STUDY（任意・30〜60分）

- 触媒×溶媒の件数・平均収率・標準偏差を表にする
- 自分なら毎回使うデータ健康診断を5項目にまとめる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. shapeの2つの数は何か
2. locの行条件と列指定はどこか
3. groupby結果に件数が必要なのはなぜか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
